|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 5:</h2>|<h1>Making It Fast<h1>|
|<h2>Section:</h2>|<h1>Incidents<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: the incident file<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

You finished Part 5. The step is captured as a graph, the sampler is one
vectorized pass, and the text streams through a detokenizer. Each of these
removes overhead, and each of them adds a way to be wrong: a graph replays
what it captured, a sampler can pick from an empty set, and a stream can show
half a character.

Each ticket gives you a **symptom** and some **evidence**. Some of the
evidence is noise. Write four lines for each ticket:

1. **Root cause.** One sentence.
2. **The number that proves it.** Not "it looks like". A computation.
3. **The fix.**
4. **The guard.** A test, an assert or an alert that catches it next time.

Four rules:

- The tickets are **not** in the order of the notebooks.
- At least one ticket is **not a bug**. "Nothing is broken" is a valid answer
  only if a number proves it.
- Write your answer **before** you open the solution.
- Every ticket has a scratch cell.

Do this section after stage 14. This notebook needs no GPU.

**The on-call colleague.** In Claude Code, type `/incident 5.1` (or any
other ticket number) to work a ticket as a conversation. The colleague has
access to the system. Ask for a log, a measurement or an experiment, and it
answers with what the system shows. When you write your four lines, it tells
you which lines are weak, and it asks a question about each one. It does not
tell you the cause until you ask for the solution.

### The reference sheet

- The model is Qwen3-1.7B, on your card, unless the ticket says otherwise.
- The vocabulary has 151,936 entries. Token 0 is `!`.
- The graph buckets are the batch sizes 1, 2, 4, 8, 16 and 32. A batch is
  padded up to the next bucket.
- UTF-8 uses 1 byte for ASCII, 2 to 3 bytes for most other scripts, and 4
  bytes for an emoji.
- The replacement character `�` (U+FFFD) is what a decoder prints for bytes
  that are not a complete character.

# Ticket 1: the graph that repeats itself

**Severity:** high. **Reported by:** the team that turned on CUDA
graphs.

> With graphs on, every answer falls apart after the first token, and
> ends in `!!!!!!`, whatever the prompt. Eager mode is fine.

**Evidence**

- Five prompts give five correct first tokens. From the second token on,
  no answer follows its prompt: `' Tokyo A A A (!!!!!!!!!!'`, `' Paris A
  A A ( (!!!!!!!!!'`. Every answer ends in a run of `!`.
- The decode step with graphs:

  ```python
  # capture, once
  static_ids = torch.zeros(1, 1, dtype=torch.long, device='cuda')
  with torch.cuda.graph(graph):
      static_logits = model(static_ids, ...).logits

  # each step
  input_ids = torch.tensor([[next_token]], device='cuda')
  graph.replay()
  next_token = static_logits[0, -1].argmax().item()
  ```

- The capture ran before the warm-up. The team thinks that the order is
  the problem.

### Solution

- **Root cause.** A graph replays the kernels with the **addresses** that
  it captured. The step makes a new tensor `input_ids` at a new address,
  and the graph never reads it. The graph reads `static_ids`, which
  still holds 0, at every step.
- **The number.** Token 0 is correct in all five answers, and token 0 is
  the only token that the eager prefill makes. The damage starts at token
  1, the first token of the graph. The answers still differ a little,
  because each cache holds its own prompt, but every answer drifts to
  `!`, which is token 0: the value in the static buffer.
- **The fix.** Copy into the static buffer:
  `static_ids.copy_(next_token_tensor)`, then `graph.replay()`. The same
  is true for the positions and the slot mapping.
- **The guard.** A test: 20 tokens with graphs must equal 20 tokens in
  eager mode. Assert that the `data_ptr()` of each graph input is the
  same at capture and at replay.

**The noise.** The order of the capture and the warm-up. It matters for
other reasons, and here it changes nothing.

**What you could ask the system**

The on-call colleague answers these questions. Each answer is a piece of evidence that the ticket does not show.

- *What does `static_ids` contain during the steps?*
  `[[0]]`, at every step.
- *What happens if we capture after the warm-up?*
  The same five identical answers.
- *Is the first token from the graph?*
  No. The first token comes from the prefill, which runs eager.

# Ticket 2: the victim is always the owner of block 0

**Severity:** critical. **Reported by:** users.

> Under load, now and then one answer turns into nonsense in the middle.
> At night, with one user at a time, it never happens.

**Evidence**

- A debug dump of 9 damaged answers: each damaged request had block 0 in
  its block table.
- The batch sizes of the steps where the damage started: 3, 5, 6, 7, 9,
  12, 3, 5, 6. The logs never show damage in a step of batch 1, 2, 4 or
  8.
- The graph runner pads a batch up to the next bucket. For a padding
  row, the [slot mapping](../../GLOSSARY.md#slot-mapping) is `0`.
- The team suspects a race in the new prefix cache.

### Solution

- **Root cause.** A padding row still runs the whole step, and the step
  writes K and V for every row. The slot of the padding rows is 0, so
  every padded step writes garbage into slot 0 of block 0. Block 0
  belongs to a real sequence.
- **The number.** All the damaged steps have a batch size that is not a
  bucket, so they have padding rows: 3 pads to 4, 5 to 8, 12 to 16. The
  sizes that are buckets never show damage. And the victim is always the
  owner of block 0.
- **The fix.** Map each padding row to slot -1, and make the write kernel
  skip a negative slot (stage 23).
- **The guard.** Fill block 0 with a known pattern, run a padded step,
  and check that the pattern did not change. Assert that the slots of
  the padding rows are -1.

**The noise.** The prefix cache. With it off, the damage continues. A
shared prefix in block 0 would make it worse, but it is not the cause.

In [ ]:
buckets = [1, 2, 4, 8, 16, 32]
for batch in (3, 5, 6, 7, 9, 12, 1, 2, 4, 8):
    bucket = next(b for b in buckets if b >= batch)
    print(f'batch {batch:2d} -> bucket {bucket:2d}: {bucket - batch} padding rows')

**What you could ask the system**

The on-call colleague answers these questions. Each answer is a piece of evidence that the ticket does not show.

- *Where does the KV write of a padding row go?*
  To slot 0, which is position 0 of block 0.
- *What happens with the prefix cache off?*
  The damage still happens, at the same rate.
- *What happens with graphs off?*
  No damage in 48 hours. Eager mode does not pad.

# Ticket 3: graphs help at batch 1 and not at batch 64

**Severity:** low. **Reported by:** the performance team.

> CUDA graphs give 1.91x at batch 1 and only 1.11x at batch 64. The
> graphs must be broken for large batches.

**Evidence**

| batch | eager, ms for each step | graph, ms for each step |
|---|---|---|
| 1 | 24.1 | 12.6 |
| 64 | 27.9 | 25.2 |

- A profile of the eager step at batch 1: the CPU needs about 24 ms to
  launch the kernels of one step. The GPU work is about 12.6 ms.

### Solution: nothing is broken

- **Root cause.** In eager mode the CPU launches the kernels, and the GPU
  runs them at the same time. The step takes about as long as the slower
  of the two. A graph removes the CPU part. At batch 1 the CPU (24 ms) is
  slower than the GPU (12.6 ms), so the graph removes the bottleneck. At
  batch 64 the GPU (25 ms) is slower than the CPU, and the graph removes
  a cost that was already hidden.
- **The number.** At batch 1: eager 24.1 ms is the CPU time, and the
  graph is the GPU time, 12.6 ms. At batch 64: eager 27.9 ms is a little
  more than the GPU time of 25 ms, and the graph gives 25.2 ms. The graph
  can only win the part where the CPU is slower.
- **The fix.** Nothing to fix. Graphs matter most at small batch, which
  is where every interactive request lives.
- **The guard.** Report the CPU time and the GPU time of the step
  separately (Nsight Systems, Part 8). A speedup is only a surprise if
  you did not know which of the two was slower.

In [ ]:
for batch, cpu, gpu, eager, graph in [(1, 24, 12.6, 24.1, 12.6), (64, 24, 25, 27.9, 25.2)]:
    print(f'batch {batch:2d}: max(CPU {cpu}, GPU {gpu}) = {max(cpu, gpu)} ms, eager {eager}, graph {graph}, '
          f'speedup {eager / graph:.2f}x')

**What you could ask the system**

The on-call colleague answers these questions. Each answer is a piece of evidence that the ticket does not show.

- *How much GPU work is in the step at batch 64?*
  About 25 ms.
- *How long does the CPU need to launch the step at batch 64?*
  About 24 ms, the same as at batch 1. It is the same number of kernels.
- *How many kernels does one step launch?*
  About 1,000.

# Ticket 4: the sampler that costs more than the model

**Severity:** medium. **Reported by:** the performance team.

> The step at batch 128 takes 79 ms. The forward pass takes 38 ms. Where
> do the other 41 ms go?

**Evidence**

- The sampler time for each step: 2.6 ms at batch 8, 20.5 ms at batch 64,
  41 ms at batch 128.
- The sampler:

  ```python
  def sample(logits, requests):
      out = []
      for row, req in zip(logits, requests):
          probs = torch.softmax(row / req.temperature, -1)
          probs = apply_top_p(probs, req.top_p)
          out.append(torch.multinomial(probs, 1).item())
      return out
  ```

- A colleague says: "The vocabulary has 151,936 entries. A sampler over
  that is slow by nature."

### Solution

- **Root cause.** The sampler is a Python loop over the requests. Each row
  launches about 11 small kernels and waits for the GPU with `.item()`.
  The cost grows with the batch.
- **The number.** 41 ms / 128 = 0.32 ms for each request, the same at
  batch 8 (2.6 / 8 = 0.33) and at batch 64 (20.5 / 64 = 0.32). A constant
  cost for each request is the fingerprint of a loop. The vectorized
  version does all 128 rows in 1.1 ms.
- **The fix.** One vectorized pass (stage 13): a tensor of temperatures,
  a tensor of top-p values, one sort, one sample, one copy to the CPU.
- **The guard.** Measure the sampler time at batch 128 in the tests, and
  fail above a few ms.

**The noise.** The size of the vocabulary. A sort of 128 x 151,936 values
on the GPU takes about 1 ms. The size is not the problem; the loop is.

In [ ]:
for batch, ms in [(8, 2.6), (64, 20.5), (128, 41)]:
    print(f'batch {batch:3d}: {ms / batch:.2f} ms per request')

**What you could ask the system**

The on-call colleague answers these questions. Each answer is a piece of evidence that the ticket does not show.

- *How long does a vectorized sampler take on the same logits at batch 128?*
  1.1 ms in the test from stage 13.
- *How many GPU-to-CPU synchronizations does one step do?*
  128. One `.item()` for each row.
- *How many kernels does the sampler launch at batch 128?*
  About 1,400.

# Ticket 5: exclamation marks when the model is sure

**Severity:** medium. **Reported by:** users.

> With `top_p=0.5`, some answers contain runs of `!!!!!!`. It happens
> more often in code.

**Evidence**

- The sampler:

  ```python
  sorted_probs, order = probs.sort(descending=True)
  cumulative = sorted_probs.cumsum(-1)
  sorted_probs[cumulative > top_p] = 0
  probs = torch.zeros_like(probs).scatter(-1, order, sorted_probs)
  token = (probs / torch.empty_like(probs).exponential_()).argmax(-1)
  ```

- The team logged 400 steps that produced `!`. In all 400 steps, the
  largest probability was above 0.5.
- Code is more predictable than prose.

### Solution

- **Root cause.** The mask removes every token whose **cumulative**
  probability is above `top_p`, and that includes the token that crosses
  the threshold. When the top token alone has more than `top_p`, the
  mask removes it too, and nothing is left. The row is all zeros, and
  `argmax` of zeros is 0, which is `!`.
- **The number.** In all 400 steps with `!`, the top probability was
  above 0.5 = `top_p`. When `sorted_probs[0] > top_p`, then
  `cumulative[0] > top_p`, and the first token is masked.
- **The fix.** Keep the token that crosses the threshold: mask where
  `cumulative - sorted_probs > top_p`. The first token then always
  survives.
- **The guard.** Assert that every row keeps at least one token. A
  distribution test (stage 13) with a very peaked distribution.

**The noise.** "More often in code". Code is predictable, so the model is
often sure. The correlation is real, and it points to the cause, but code
is not the cause.

In [ ]:
import torch
probs = torch.tensor([0.7, 0.2, 0.1])
cumulative = probs.cumsum(-1)
print('buggy mask keeps: ', probs[~(cumulative > 0.5)])
print('correct mask keeps:', probs[~(cumulative - probs > 0.5)])

**What you could ask the system**

The on-call colleague answers these questions. Each answer is a piece of evidence that the ticket does not show.

- *What does `probs` contain after the mask, in one of those steps?*
  Only zeros.
- *What does `argmax` return for a row of zeros?*
  `0`.
- *Does it happen with `top_p=0.95`?*
  Yes, but more rarely. Only when the largest probability is above 0.95.

# Ticket 6: the seed that works only at night

**Severity:** medium. **Reported by:** a customer.

> I send `seed=42` and temperature 0.8. At night I get the same answer
> every time. During the day I get a different answer every time.

**Evidence**

- The sampler:

  ```python
  for req in batch:
      if req.seed is not None:
          torch.manual_seed(req.seed)
  tokens = torch.multinomial(probs, 1)      # the whole batch
  ```

- At night the request runs alone. During the day it shares a batch with
  20 to 60 other requests.
- An engineer says: "It is the bf16 batch effect of Part 1."

### Solution

- **Root cause.** The sampler seeds the **global** generator, and then
  samples the whole batch from it. The random number that row 5 gets
  depends on how many rows come before it, and on the seeds of the
  other requests. The seed of one request controls nothing when others
  share the generator.
- **The number.** At temperature 0, the answers are identical during the
  day, so the arithmetic is stable. The first different token had
  probability 0.12 against 0.41. A rounding difference can flip a near
  tie, not a gap of 0.29. The difference comes from the random numbers.
- **The fix.** One generator for each request, seeded with the seed of
  the request, and advanced only by that request. In JAX, one key for
  each request (stage 13).
- **The guard.** A test: the same seeded request, in two different
  batches, must give the same tokens.

**The noise.** The bf16 theory. It is a real effect, and the temperature
0 test rules it out.

**What you could ask the system**

The on-call colleague answers these questions. Each answer is a piece of evidence that the ticket does not show.

- *At temperature 0, during the day, are the answers the same?*
  Yes. Ten answers are identical.
- *At the first different token, how close are the top two probabilities?*
  Not close: 0.41 and 0.12. The answer took the token with 0.12.
- *What if the request is the only one with a seed in the batch?*
  The answers still differ, because the other rows use random numbers from the same generator.

# Ticket 7: the melting face that melts

**Severity:** low. The users post screenshots. **Reported by:** the
front-end team.

> In the stream, emoji and some rare characters show as `���`. When the
> answer is complete and the page reloads, the text is correct.

**Evidence**

- The stream code:

  ```python
  for token in generated_tokens():
      yield tokenizer.decode([token])
  ```

- The emoji 🫠 appears as `���` in the stream: three replacement
  characters.
- The emoji 🙂 streams correctly.
- The front-end team changed the font of the chat last week.

### Solution

- **Root cause.** The tokenizer works on bytes. 🫠 is 4 bytes in UTF-8,
  and the tokenizer splits those bytes into 3 tokens. Each token alone is
  not a complete character, so each `decode([token])` prints `�`.
- **The number.** 🫠 is 3 tokens, and the stream shows 3 replacement
  characters. 🙂 is 1 token, and it streams correctly. The three tokens
  together decode to 🫠.
- **The fix.** Incremental detokenization (stage 14): decode a window of
  the last tokens, and emit only the new text when it does not end in
  an incomplete character.
- **The guard.** A fuzz test: for random token sequences, the streamed
  text must equal the decode of all the tokens at once.

**The noise.** The font. A font cannot turn a correct character into
three replacement characters, and the text is correct after a reload.

In [ ]:
emoji = '🫠'
print(len(emoji.encode('utf-8')), 'bytes:', emoji.encode('utf-8'))
print('two bytes alone:', emoji.encode('utf-8')[:2].decode('utf-8', errors='replace'))

**What you could ask the system**

The on-call colleague answers these questions. Each answer is a piece of evidence that the ticket does not show.

- *How many tokens does the tokenizer make for 🫠?*
  3 tokens, `[9284, 104, 254]`.
- *How many tokens does it make for 🙂?*
  1 token, `[145080]`.
- *What does `tokenizer.decode([9284, 104, 254])` give?*
  🫠

# Ticket 8: the words that stick together

**Severity:** medium. **Reported by:** users, after a model change.

> Since we moved from Qwen3 to Mistral-7B, the streamed text has no
> spaces: `ThecapitalofFranceisParis.` The final answer in the history is
> correct.

**Evidence**

- The stream code is the same as in Ticket 7: `tokenizer.decode([token])`
  for each token.
- Qwen3 uses a byte-level BPE tokenizer. Mistral-7B uses a SentencePiece
  tokenizer, which marks a space with `▁` at the start of a word.
- The stream has 0 spaces in the example. The final text has 5.

### Solution

- **Root cause.** A SentencePiece decoder removes the space at the start
  of the text, because the first word of a text has no space before it.
  When you decode each token alone, every token is the start of a text,
  so every space goes away.
- **The number.** The stream has 0 spaces and the final text has 5. Six
  tokens start with `▁`. The first one is at the start of the text, so
  its space goes away in both cases. The other 5 lose their space only
  in the stream. Qwen3 marks spaces as bytes, and it keeps them.
- **The fix.** The same incremental detokenizer as in Ticket 7. Decode
  the previous tokens and the new ones together, and emit only the
  difference.
- **The guard.** The same fuzz test, run with each tokenizer that the
  server supports.

Tickets 7 and 8 have one cause: a token is not a unit of text.

**What you could ask the system**

The on-call colleague answers these questions. Each answer is a piece of evidence that the ticket does not show.

- *What does `tokenizer.convert_ids_to_tokens` show for the Mistral tokens?*
  `['▁The', '▁capital', '▁of', '▁France', '▁is', '▁Paris', '.']`
- *What does `tokenizer.decode` give for the single token `▁capital`?*
  `'capital'`, with no space.
- *What does `decode` give for the two tokens `▁The` and `▁capital` together?*
  `'The capital'`.

# Ticket 9: the stop string that does not stop

**Severity:** medium. **Reported by:** a customer.

> We send `stop=["###"]`. Most answers stop there. Some answers show
> `###` and then continue with the next section.

**Evidence**

- The stop check:

  ```python
  text = tokenizer.decode([token])
  if any(s in text for s in request.stop):
      finish(request)
  ```

- In the answers that stop correctly, the log shows the token 14374,
  which is `###`.
- In the answers that do not stop, the log shows the token 565 (`##`)
  and then the token 2 (`#`).
- The customer thinks that the model ignores the stop instruction
  sometimes.

### Solution

- **Root cause.** The check looks inside one token. When the model writes
  `###` as the two tokens `##` and `#`, no single token contains the stop
  string.
- **The number.** 100% of the leaks have the stop string across a token
  boundary, and 0% of the leaks have it in one token.
- **The fix.** Check the end of the whole text, not the last token.
  Also hold back the end of the text that could be the start of a stop
  string, so that the stream never shows `##` before a stop (stage 14).
- **The guard.** A test that forces the stop string across a token
  boundary: `##` + `#`, and `#` + `##`.

**The noise.** "The model ignores the instruction". The stop string is
an instruction to the server, not to the model. The model wrote `###`
correctly.

**What you could ask the system**

The on-call colleague answers these questions. Each answer is a piece of evidence that the ticket does not show.

- *How many of the leaks have `###` inside one token?*
  None. In every leak, `###` spans two tokens.
- *Does the stream show `##` before a correct stop?*
  Sometimes. When a later token completes the stop string, the stream already showed part of it.

### The pattern in the tickets

| The shape of the number | What it usually means | Tickets |
|---|---|---|
| Outputs that do not depend on the input | A graph that reads a stale buffer | 1 |
| Damage only at the sizes that need padding | The padding writes somewhere real | 2 |
| The speedup equals the part that was the bottleneck | Nothing is broken | 3 |
| A constant cost for each request | A Python loop | 4 |
| A failure exactly above a threshold of the input | An empty set, or an overflow | 5 |
| A difference with a large probability gap | Random numbers, not rounding | 6 |
| The count of broken characters equals the count of tokens | Decoding one token at a time | 7, 8, 9 |